## Lab 04 - ANN DL - Optimizers effect on Classification Model 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.datasets import make_regression
from sklearn import metrics
from sklearn import decomposition
from sklearn import manifold
from tqdm.notebook import trange, tqdm

#TensorFlow and Keras Libraries
import tensorflow as tf
from tensorflow.keras.layers import Dense, Flatten, Conv2D
from tensorflow.keras import Model
import torch
from torchvision import transforms
from tensorflow.keras.utils import to_categorical
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.datasets as datasets
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# Hiding the warnings
import warnings
warnings.filterwarnings('ignore')

# Extra Libraries
import copy
import random
import time

In [2]:
data_path = "DataBase/Class_Data/house-prices.csv"
df = pd.read_csv(data_path)

In [3]:
df.head()

,Home,Price,SqFt,Bedrooms,Bathrooms,Offers,Brick,Neighborhood
0,1,114300,1790,2,2,2,No,East
1,2,114200,2030,4,2,3,No,East
2,3,114800,1740,3,2,1,No,East
3,4,94700,1980,3,2,3,No,East
4,5,119800,2130,3,3,3,No,East


In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Home,128.0,64.500000,37.094474,1.0,32.75,64.5,96.25,128.0
Price,128.0,130427.343750,26868.770371,69100.0,111325.00,125950.0,148250.00,211200.0
SqFt,128.0,2000.937500,211.572431,1450.0,1880.00,2000.0,2140.00,2590.0
Bedrooms,128.0,3.023438,0.725951,2.0,3.00,3.0,3.00,5.0
Bathrooms,128.0,2.445312,0.514492,2.0,2.00,2.0,3.00,4.0
Offers,128.0,2.578125,1.069324,1.0,2.00,3.0,3.00,6.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 128 entries, 0 to 127
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Home          128 non-null    int64 
 1   Price         128 non-null    int64 
 2   SqFt          128 non-null    int64 
 3   Bedrooms      128 non-null    int64 
 4   Bathrooms     128 non-null    int64 
 5   Offers        128 non-null    int64 
 6   Brick         128 non-null    object
 7   Neighborhood  128 non-null    object
dtypes: int64(6), object(2)
memory usage: 8.1+ KB


In [6]:
df.isna().sum()

Home            0
Price           0
SqFt            0
Bedrooms        0
Bathrooms       0
Offers          0
Brick           0
Neighborhood    0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
cat_col = []
for i in df.columns:
    if df[i].dtype == 'O':
        cat_col.append(i)

cat_col

['Brick', 'Neighborhood']

In [9]:
df['Brick'].value_counts()

Brick
No     86
Yes    42
Name: count, dtype: int64

In [10]:
df[cat_col].nunique()

Brick           2
Neighborhood    3
dtype: int64

In [11]:
# Now using Label Encoder :
le1 = LabelEncoder()
df['Brick'] = le1.fit_transform(df['Brick'])
df['Brick'].unique()

le2 = LabelEncoder()
df['Neighborhood'] = le2.fit_transform(df['Neighborhood'])
df['Neighborhood'].unique()

array([0, 1, 2])

In [12]:
# Dropping column - "Home"
df.drop('Home',axis=1,inplace=True)

In [13]:
# Standardization of data :
sc1 = MinMaxScaler()
df[['Price']] = sc1.fit_transform(df[['Price']])

sc2 = MinMaxScaler()
df[['SqFt']] = sc2.fit_transform(df[['SqFt']])

In [14]:
df['Price'] = round(df['Price'],2)
df['SqFt'] = round(df['SqFt'],2)

In [15]:
df.head()

,Price,SqFt,Bedrooms,Bathrooms,Offers,Brick,Neighborhood
0,0.32,0.30,2,2,2,0,0
1,0.32,0.51,4,2,3,0,0
2,0.32,0.25,3,2,1,0,0
3,0.18,0.46,3,2,3,0,0
4,0.36,0.60,3,3,3,0,0


In [16]:
# Binary Class Classification :
X = df.drop('Brick',axis=1)
y = df['Brick']

In [17]:
# Train-Test Split :
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.33,random_state=42)
print("X_train shape : ",X_train.shape)
print("X_test shape : ",X_test.shape)
print("y_train shape : ",y_train.shape)
print("y_test shape : ",y_test.shape)

X_train shape :  (85, 6)
X_test shape :  (43, 6)
y_train shape :  (85,)
y_test shape :  (43,)


In [18]:
# Building MLP model for Binary Class Classification :
model = tf.keras.models.Sequential()
model.add(tf.keras.layers.Dense(45,activation='relu',input_dim = 6))
model.add(tf.keras.layers.Dense(25,activation='relu'))
model.add(tf.keras.layers.Dense(15,activation='relu'))
model.add(tf.keras.layers.Dense(1,activation='sigmoid'))

In [19]:
# 1. We are using adam optimizer :
model.compile(loss=tf.keras.losses.BinaryCrossentropy(), optimizer='adam', metrics=['BinaryAccuracy'])

start_train = time.time()
history = model.fit(X_train, y_train, epochs=500)
end_train = time.time()

training_time = end_train - start_train
print(f"Training time: {training_time:.2f} seconds")

Epoch 1/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - BinaryAccuracy: 0.2916 - loss: 0.7442 
Epoch 2/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 0.3150 - loss: 0.7239
Epoch 3/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.3679 - loss: 0.7068 
Epoch 4/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 0.4756 - loss: 0.6898
Epoch 5/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.7338 - loss: 0.6708 
Epoch 6/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.6909 - loss: 0.6539 
Epoch 7/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.6987 - loss: 0.6452 
Epoch 8/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 0.7065 - loss: 0.6174
Epoch 9/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.7377 - loss: 0.5932 
Epoch 10/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - BinaryAccuracy: 0.7416 - loss: 0.5873 
Epoch 11/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - BinaryAccuracy: 0.7065 - loss: 0.6115 
Epoch 1

In [20]:
start_test = time.time()
test_loss, test_acc = model.evaluate(X_test, y_test)
end_test = time.time()
testing_time = end_test - start_test
print(f"Testing time: {testing_time:.2f} seconds")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - BinaryAccuracy: 0.7357 - loss: 0.6430
Testing time: 0.14 seconds


In [21]:
y_pred = model.predict(X_test)
print(y_pred.shape)
y_bi_pred = (y_pred > 0.5).astype(int).flatten()
print(y_bi_pred.shape)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
(43, 1)
(43,)


In [22]:
# Calculating Accuracy, precision, recall, f1-score and Confusion Matrix
accuracy = accuracy_score(y_test, y_bi_pred)
conf_matrix = confusion_matrix(y_test, y_bi_pred)
class_report = classification_report(y_test, y_bi_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", class_report)

Accuracy: 0.7441860465116279
Confusion Matrix:
 [[20  5]
 [ 6 12]]
Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.80      0.78        25
           1       0.71      0.67      0.69        18

    accuracy                           0.74        43
   macro avg       0.74      0.73      0.74        43
weighted avg       0.74      0.74      0.74        43



In [23]:
# 2. We are using rmsprop optimizer :
model.compile(loss=tf.keras.losses.BinaryCrossentropy(), optimizer='rmsprop', metrics=['BinaryAccuracy'])

start_train = time.time()
history = model.fit(X_train, y_train, epochs=500)
end_train = time.time()

training_time = end_train - start_train
print(f"Training time: {training_time:.2f} seconds")

Epoch 1/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - BinaryAccuracy: 0.8747 - loss: 0.3351  
Epoch 2/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - BinaryAccuracy: 0.9374 - loss: 0.1210 
Epoch 3/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - BinaryAccuracy: 0.9530 - loss: 0.1019 
Epoch 4/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - BinaryAccuracy: 0.9354 - loss: 0.1067 
Epoch 5/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - BinaryAccuracy: 0.9511 - loss: 0.1008 
Epoch 6/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9569 - loss: 0.1175 
Epoch 7/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - BinaryAccuracy: 0.9628 - loss: 0.1191 
Epoch 8/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - BinaryAccuracy: 0.9511 - loss: 0.0927
Epoch 9/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - BinaryAccuracy: 0.9628 - loss: 0.1114 
Epoch 10/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - BinaryAccuracy: 0.9393 - loss: 0.1014 
Epoch 11/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - BinaryAccuracy: 0.9198 - loss: 0.1831 
Epoch 1

In [24]:
start_test = time.time()
test_loss, test_acc = model.evaluate(X_test, y_test)
end_test = time.time()
testing_time = end_test - start_test
print(f"Testing time: {testing_time:.2f} seconds")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - BinaryAccuracy: 0.7720 - loss: 0.9309
Testing time: 0.15 seconds


In [25]:
y_pred = model.predict(X_test)
print(y_pred.shape)
y_bi_pred = (y_pred > 0.5).astype(int).flatten()
print(y_bi_pred.shape)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
(43, 1)
(43,)


In [26]:
# Calculating Accuracy, precision, recall, f1-score and Confusion Matrix
accuracy = accuracy_score(y_test, y_bi_pred)
conf_matrix = confusion_matrix(y_test, y_bi_pred)
class_report = classification_report(y_test, y_bi_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", class_report)

Accuracy: 0.7674418604651163
Confusion Matrix:
 [[22  3]
 [ 7 11]]
Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.88      0.81        25
           1       0.79      0.61      0.69        18

    accuracy                           0.77        43
   macro avg       0.77      0.75      0.75        43
weighted avg       0.77      0.77      0.76        43



In [27]:
# 3. We are using adagrad optimizer :
model.compile(loss=tf.keras.losses.BinaryCrossentropy(), optimizer='adagrad', metrics=['BinaryAccuracy'])

start_train = time.time()
history = model.fit(X_train, y_train, epochs=500)
end_train = time.time()

training_time = end_train - start_train
print(f"Training time: {training_time:.2f} seconds")

Epoch 1/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9765 - loss: 0.0511  
Epoch 2/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 1.0000 - loss: 0.0458
Epoch 3/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9902 - loss: 0.0431 
Epoch 4/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9824 - loss: 0.0367 
Epoch 5/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9941 - loss: 0.0265 
Epoch 6/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9902 - loss: 0.0322 
Epoch 7/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9902 - loss: 0.0321 
Epoch 8/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9902 - loss: 0.0322 
Epoch 9/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 0.9824 - loss: 0.0412
Epoch 10/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - BinaryAccuracy: 0.9941 - loss: 0.0336 
Epoch 11/500
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - BinaryAccuracy: 0.9902 - loss: 0.0366
Epoch 1

In [28]:
start_test = time.time()
test_loss, test_acc = model.evaluate(X_test, y_test)
end_test = time.time()
testing_time = end_test - start_test
print(f"Testing time: {testing_time:.2f} seconds")

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - BinaryAccuracy: 0.7202 - loss: 0.8651
Testing time: 0.16 seconds


In [29]:
y_pred = model.predict(X_test)
print(y_pred.shape)
y_bi_pred = (y_pred > 0.5).astype(int).flatten()
print(y_bi_pred.shape)

1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/stepWARNING:tensorflow:6 out of the last 6 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x0000029EE2350EA0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
(43, 1)
(43,)


In [30]:
# Calculating Accuracy, precision, recall, f1-score and Confusion Matrix
accuracy = accuracy_score(y_test, y_bi_pred)
conf_matrix = confusion_matrix(y_test, y_bi_pred)
class_report = classification_report(y_test, y_bi_pred)

print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", class_report)

Accuracy: 0.7209302325581395
Confusion Matrix:
 [[20  5]
 [ 7 11]]
Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.80      0.77        25
           1       0.69      0.61      0.65        18

    accuracy                           0.72        43
   macro avg       0.71      0.71      0.71        43
weighted avg       0.72      0.72      0.72        43

